# P06: Applications & Modern Python

Classes let you bundle data with the operations that belong to it -- a natural fit
for a `Participant`, a `Trial`, or a `Stimulus`. Modern Python's `@dataclass`
removes almost all of the boilerplate, and it also steers you away from a subtle
class-level bug.

## A `Participant` as a dataclass

In [1]:
from dataclasses import dataclass, field

@dataclass
class Participant:
    subject_id: int
    age: int
    rts: list = field(default_factory=list)   # each participant gets their OWN list

p1 = Participant(1, 22)
p2 = Participant(2, 31)
p1.rts.append(512)
print(p1.rts, p2.rts)    # [512] []  -- independent, exactly what we want
print(p1)                # dataclasses give you a readable repr for free

[512] []
Participant(subject_id=1, age=22, rts=[512])


## The shared-class-attribute trap

In [2]:
# the SAME hazard as P05's mutable default, now at the class level
class BadParticipant:
    rts = []                       # <-- one list shared by ALL instances!
    def __init__(self, subject_id):
        self.subject_id = subject_id

a = BadParticipant(1)
b = BadParticipant(2)
a.rts.append(512)
print("b.rts:", b.rts)             # [512] -- b sees a's data!

b.rts: [512]


:::{admonition} Human check — class attributes vs instance attributes
:class: warning
Defining `rts = []` in the class body (rather than assigning `self.rts = []` in
`__init__`, or using `field(default_factory=list)`) creates a single list shared by
every instance. Each new participant appears to inherit the previous participant's
trials. The code runs and looks reasonable in a one-object test -- the bug only
appears once you create a second participant. Prefer `@dataclass` with
`default_factory`, which makes the right thing the easy thing.
:::

:::{admonition} Modern Python
:class: tip
`@dataclass` (Python 3.7+) auto-generates `__init__`, `__repr__`, and `__eq__`.
For analysis code it is usually all the class machinery you need, and it is far
less error-prone than writing those methods by hand.
:::